In [1]:
import fitz  
import cv2
import numpy as np
import os

print("[+] Starting Universal PDF Pre-processing Pipeline...")

pdf_name = "كشف حساب الراجحي.pdf"

if not os.path.exists(pdf_name):
    print(f"[!] Error: Cannot find '{pdf_name}' in the current folder!")
else:
    doc = fitz.open(pdf_name)
    page = doc.load_page(0)

    try:
        fitz.tools.set_aa_level(0)
    except Exception:
        pass

    optimal_zoom = 5.0
    pix = None

    while optimal_zoom >= 3.0:
        matrix = fitz.Matrix(optimal_zoom, optimal_zoom)
        try:
            pix = page.get_pixmap(matrix=matrix, alpha=False)
            break
        except Exception:
            print(f"[!] RAM allocation failed at {optimal_zoom}x zoom. Stepping down...")
            optimal_zoom -= 0.5

    if pix is None:
        print("[!] Emergency fallback to 2.0x zoom.")
        try:
            matrix = fitz.Matrix(2.0, 2.0)
            pix = page.get_pixmap(matrix=matrix, alpha=False)
            optimal_zoom = 2.0
        except Exception as e:
            print(f"[!] Critical Error! {e}")

    if pix is not None:
        print(f"[✓] Canvas successfully rendered at {optimal_zoom:.1f}x zoom.")

        img_data = np.frombuffer(pix.samples, dtype=np.uint8).reshape((pix.h, pix.w, pix.n))
        img_gray = cv2.cvtColor(img_data, cv2.COLOR_RGB2GRAY)

        print("[+] Applying Bilateral Filter (Edge-Preserving Smoothness)...")
        img_filtered = cv2.bilateralFilter(img_gray, d=9, sigmaColor=75, sigmaSpace=75)

        print("[+] Calculating mathematically optimal threshold via Otsu's Binarization...")
        optimal_thresh, binary_1bit = cv2.threshold(
            img_filtered, 
            0, 
            255, 
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )

        print(f"[✓] Optimal dynamic threshold calculated: {optimal_thresh}")

        output_path = "page_1bit_universal.tiff"
        cv2.imwrite(output_path, binary_1bit)
        
        print(f"[✓] Flawless Universal File generated and saved to: {output_path}")

[+] Starting Universal PDF Pre-processing Pipeline...
[✓] Canvas successfully rendered at 5.0x zoom.
[+] Applying Bilateral Filter (Edge-Preserving Smoothness)...
[+] Calculating mathematically optimal threshold via Otsu's Binarization...
[✓] Optimal dynamic threshold calculated: 173.0
[✓] Flawless Universal File generated and saved to: page_1bit_universal.tiff
